# GraphKnows Service Demo

**Step-by-step walkthrough of how GraphKnows stores, retrieves, and scores information from real question-answering benchmarks.**

GraphKnows ingests conversational and document data, builds a knowledge graph, and retrieves the most relevant memories when a question is asked. This notebook runs the full pipeline on real benchmark cases and measures how well each stage performs.

**Sections:**
1. Health check — confirm all backends (Memgraph, Neo4j, pgvector) are reachable
2. Evaluation cases — 3 hard LoCoMo cases (multi-session memory) and 3 BEAM cases (document recall)
3. Entity extraction — how GraphKnows reads text and finds people, organizations, and relationships
4. Ontology routing — automatic schema selection based on the content type
5. LLM-assisted extraction — using an LLM to discover deeper relationships the rule-based extractor misses
6. Short-term memory (STM) — ingest all 6 cases and measure immediate recall quality
7. Long-term memory (LTM) — promote to the graph store and run the DETRetriever
8. Benchmark comparison — load and compare saved full-run result files
9. Save trace

## Runtime Selectors

In [1]:
# LLM model for DSPy extraction and topic summarization (OpenRouter).
LLM_MODEL = 'moonshotai/kimi-k2'

# Run GDS (PageRank + Leiden) during LTM flush. Requires the GDS plugin.
RUN_GDS = True

# Write to Memgraph / pgvector / Neo4j.
RUN_STM_LTM_WRITEBACK = True

# Run DSPy LLM-assisted extraction (requires OPENROUTER_API_KEY in .env).
RUN_DSPY_LLM_ASSISTED = True

# Which LoCoMo and BEAM case to flush to LTM for the deep retrieval demo.
LTM_DEMO_CASE_ID = 'locomo-smoke-006'

print({
    'LLM_MODEL': LLM_MODEL,
    'RUN_GDS': RUN_GDS,
    'RUN_STM_LTM_WRITEBACK': RUN_STM_LTM_WRITEBACK,
    'RUN_DSPY_LLM_ASSISTED': RUN_DSPY_LLM_ASSISTED,
    'LTM_DEMO_CASE_ID': LTM_DEMO_CASE_ID,
})

{'LLM_MODEL': 'moonshotai/kimi-k2', 'RUN_GDS': True, 'RUN_STM_LTM_WRITEBACK': True, 'RUN_DSPY_LLM_ASSISTED': True, 'LTM_DEMO_CASE_ID': 'locomo-smoke-006'}


## Section 0 — Setup & Health Check

Loads the `graphknows` package directly from the working tree (no install needed), reads credentials from `.env`, and confirms that all three backend stores are reachable before any data flows.

In [2]:
from __future__ import annotations

import importlib
import json
import os
import re
import sys
import time
import uuid
from pathlib import Path
from typing import Any

# --- Repo root and forced reload from working tree ---
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'graphknows').exists():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'graphknows').exists():
    raise RuntimeError('Run from the repo root or research/.')
sys.path.insert(0, str(REPO_ROOT))

importlib.invalidate_caches()
for _mod in [k for k in sys.modules if k == 'graphknows' or k.startswith('graphknows.')]:
    del sys.modules[_mod]

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / '.env')

try:
    from yfiles_jupyter_graphs import GraphWidget
except Exception:
    GraphWidget = None

# --- GraphKnows microservice imports ---
from graphknows.config import GraphKnowsSettings
from graphknows.config.health import check_backends, get_memory_stats, get_version
from graphknows.config.adapters.gliner_model import load_gliner2_model
from graphknows.config.builders.stores import build_promotion_pipeline, build_stm_service
from graphknows.container import GraphKnowsContainer
from graphknows.graphgen.extraction.relations import (
    DSPyExtractor,
    GraphExtractorModule,
    _OPEN_DOMAIN_CLASSES,
    _OPEN_DOMAIN_RELATION_TYPES,
)
from graphknows.ingestion.ontology.loader import OntologyLoader
from graphknows.ingestion.ontology.router import OntologyRouter
from graphknows.memory.extractor import GLiNER2EntityExtractor, _SpacySchemaMiner
from graphknows.memory.models.promotion import PromotionRequest
from graphknows.memory.query.retriever import DETRetriever

# --- Display helper ---
def show(obj: Any) -> None:
    try:
        from IPython.display import JSON, display
        display(JSON(json.loads(json.dumps(obj, default=str))))
    except Exception:
        print(json.dumps(obj, indent=2, default=str))

# --- Trace accumulator ---
run_id = f'demo-{uuid.uuid4().hex[:8]}'
artifact_dir = REPO_ROOT / 'research' / 'notebook_artifacts' / run_id
artifact_dir.mkdir(parents=True, exist_ok=True)
trace: dict[str, Any] = {'run_id': run_id, 'artifact_dir': str(artifact_dir), 'components': {}}

# --- Settings ---
settings = GraphKnowsSettings()
settings.summarization_model = LLM_MODEL

print('graphknows source root :', REPO_ROOT / 'graphknows')
print('run_id                 :', run_id)
print('git SHA                :', get_version())
print('LLM model              :', LLM_MODEL)
print('embed model            :', settings.embed_model)

graphknows source root : /app/graphknows
run_id                 : demo-0c6343ea
git SHA                : ea20dbeb813a
LLM model              : moonshotai/kimi-k2
embed model            : openai/text-embedding-3-small


In [ ]:
# ---------------------------------------------------------------------------
# Shared yfiles graph visualisation helper
# ---------------------------------------------------------------------------

# Node colour palette — one colour per entity / node type.
_TYPE_PALETTE: dict[str, str] = {
    'PERSON':         '#4A90D9',   # blue
    'ORG':            '#E8891A',   # orange
    'ORGANIZATION':   '#E8891A',
    'PLACE':          '#27AE60',   # green
    'LOCATION':       '#27AE60',
    'GPE':            '#27AE60',
    'DATE':           '#8E44AD',   # purple
    'TIME':           '#8E44AD',
    'EVENT':          '#C0392B',   # red
    'PRODUCT':        '#16A085',   # teal
    'WORK':           '#2980B9',   # steel blue
    'CONCEPT':        '#7D3C98',   # violet
    'ENTITY_CONCEPT': '#7D3C98',
    'TOPIC':          '#F39C12',   # amber — LTM topic cluster nodes
    'ENTITY':         '#7F8C8D',   # grey fallback
}

def _node_color(node: dict) -> str:
    t = (node.get('type') or node.get('properties', {}).get('type') or 'ENTITY').upper()
    return _TYPE_PALETTE.get(t, _TYPE_PALETTE['ENTITY'])

def _node_size(node: dict) -> list[float]:
    t = (node.get('type') or '').upper()
    pr = float((node.get('properties') or {}).get('pagerank') or 0)
    base = 90.0 if t == 'TOPIC' else 58.0
    side = base + min(pr * 600, 38)
    return [side, side]


def _plot_graph(
    nodes: list[dict],
    edges: list[dict],
    *,
    directed: bool = True,
    layout: str = 'organic',
    title: str = '',
    height: str = '640px',
) -> Any:
    """Render a polished yfiles GraphWidget.

    nodes — list of dicts: {id, label, type, properties: {…}}
    edges — list of dicts: {id, start, end, label, properties: {…}}
    All node metadata in `properties` is shown in the sidebar on click.
    Returns the GraphWidget or None when yfiles is not installed.
    """
    if GraphWidget is None:
        print(f'{title or "graph"}: {len(nodes)} nodes, {len(edges)} edges'
              '  (install yfiles-jupyter-graphs to visualise)')
        return None

    from IPython.display import display as _display
    from ipywidgets import Layout

    w = GraphWidget(
        widget_layout=Layout(height=height, width='100%'),
        overview_enabled=True,
    )
    w.directed = directed
    w.nodes = nodes
    w.edges = edges

    # Canvas labels
    w.set_node_label_mapping(lambda i, n: n.get('label') or str(n.get('id', '')))
    w.set_edge_label_mapping(lambda i, e: e.get('label', ''))

    # Visual encoding
    w.set_node_color_mapping(lambda i, n: _node_color(n))
    w.set_node_size_mapping(lambda i, n: _node_size(n))

    # Sidebar / tooltip metadata (all non-null fields from properties)
    w.set_node_property_mapping(lambda i, n: {
        'type': n.get('type', 'ENTITY'),
        **{k: v for k, v in (n.get('properties') or {}).items() if v is not None},
    })
    w.set_edge_property_mapping(lambda i, e: e.get('properties') or {})

    # Edge width from weight or confidence score
    def _thickness(i: int, e: dict) -> float:
        p = e.get('properties') or {}
        raw = p.get('weight') or p.get('score') or p.get('confidence') or 1.0
        return float(min(max(float(raw), 0.5), 5.0))
    w.set_edge_thickness_factor_mapping(_thickness)

    # Auto-layout
    _LAYOUTS = {
        'organic':    w.organic_layout,
        'hierarchic': w.hierarchic_layout,
        'radial':     w.radial_layout,
        'circular':   w.circular_layout,
        'tree':       w.tree_layout,
    }
    _LAYOUTS.get(layout, w.organic_layout)()

    w.set_sidebar(enabled=True)

    if title:
        print(f'▶  {title}  ·  {len(nodes)} nodes  ·  {len(edges)} edges')
    _display(w)
    return w


print('Graph visualisation helper ready.')

### Backend Connectivity & Memory Stats

In [4]:
health = await check_backends(settings)
trace['health'] = health
show(health)

memory_stats_before = await get_memory_stats(settings)
trace['memory_stats_before'] = memory_stats_before
show({'memory_stats_before': memory_stats_before})

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

## Section 1 — Evaluation Cases

**LoCoMo** cases test multi-hop memory: the answer is spread across 8 separate conversation sessions and requires connecting dots between them. **BEAM** cases test document recall: the answer is buried in one of several source documents that sometimes contradict each other.

Each case loads from the official benchmark fixture files — the same files the `python -m evaluation` harness uses — so the scores here are directly comparable to a full run.

In [5]:
FIXTURE_ROOT = REPO_ROOT / 'evaluation' / 'fixtures'

locomo_all = [json.loads(l) for l in (FIXTURE_ROOT / 'locomo' / 'questions.jsonl').read_text().splitlines() if l.strip()]
beam_all   = [json.loads(l) for l in (FIXTURE_ROOT / 'beam' / '10m' / 'questions.jsonl').read_text().splitlines() if l.strip()]

LOCOMO_CASE_IDS = ['locomo-smoke-006', 'locomo-smoke-007', 'locomo-smoke-008']
BEAM_CASE_IDS   = ['beam-10m-smoke-001', 'beam-10m-smoke-003', 'beam-10m-smoke-005']

locomo_cases = {q['id']: q for q in locomo_all if q['id'] in LOCOMO_CASE_IDS}
beam_cases   = {q['id']: q for q in beam_all   if q['id'] in BEAM_CASE_IDS}

def _serialize_session(session: dict) -> str:
    """Serialize one session's messages to plain text (same logic as eval harness)."""
    lines = []
    for msg in session.get('messages', []):
        role = msg.get('role', 'message')
        content = msg.get('content', '').strip()
        if content:
            lines.append(f'{role}: {content}')
    return '\n'.join(lines)

def _case_documents(row: dict, max_docs: int = 25) -> list[dict[str, str]]:
    """Build the ordered list of (title, text) documents for a case."""
    docs: list[dict[str, str]] = []
    # BEAM / document-style cases
    for doc in row.get('documents', [])[:max_docs]:
        title = doc.get('title', 'document')
        text  = doc.get('text', '').strip()
        if text:
            docs.append({'title': title, 'text': text})
    # LoCoMo / session-style cases
    for i, sess in enumerate(row.get('sessions', [])[:max_docs]):
        title = sess.get('session_id', f'session-{i}')
        text  = _serialize_session(sess)
        if text:
            docs.append({'title': title, 'text': text})
    return docs

# Summary table
rows_summary = []
for qid in LOCOMO_CASE_IDS + BEAM_CASE_IDS:
    q = locomo_cases.get(qid) or beam_cases.get(qid)
    docs = _case_documents(q)
    rows_summary.append({
        'id': qid,
        'benchmark': 'locomo' if qid.startswith('locomo') else 'beam',
        'category': q.get('category', q.get('question_type', '?')),
        'question': q['question'],
        'expected_answer': str(q['answer']),
        'source_documents': len(docs),
    })

trace['eval_cases'] = rows_summary
show(rows_summary)

<IPython.core.display.JSON object>

## Section 2 — Pipeline A: LLM-Free Extraction (GLiNER2 + spaCy)

The demo extraction case is defined inline below — a richer, entity-dense conversation than the smoke fixtures. It has **three reasoning hops** between sessions, a deliberate red herring (a second fund that manages *other* divisions), and enough named entities to make the two extractor graphs visibly different from each other.

The LLM-free pipeline runs entirely offline:
1. **spaCy** reads the text and emits candidate entity types as schema hints
2. **GLiNER2** (DeBERTa-based) uses those hints to extract typed entity spans and binary relation triplets in a single forward pass
3. The result is a normalized `ExtractionResult(entities, relations)` that feeds into the graph store

In [6]:
class RecordingSchemaMiner:
    def __init__(self, inner: _SpacySchemaMiner) -> None:
        self.inner = inner
        self.calls: list[dict[str, Any]] = []

    def build(self, text: str) -> Any:
        schema = self.inner.build(text)
        self.calls.append({
            'component': '_SpacySchemaMiner.build',
            'input': {'text': text},
            'output': {
                'entity_spec_passed_to_gliner2': schema.entity_spec,
                'relation_spec_passed_to_gliner2': schema.relation_spec,
                'gliner_label_to_graphknows_type': schema.label_to_type,
            },
        })
        return schema


class RecordingSchema:
    def __init__(self, inner: Any, calls: list[dict[str, Any]]) -> None:
        self.inner = inner
        self.calls = calls

    def entities(self, entity_spec: dict[str, Any]) -> Any:
        self.calls.append({'method': 'schema.entities', 'input': entity_spec})
        return self.inner.entities(entity_spec)

    def relations(self, relation_spec: dict[str, str]) -> Any:
        self.calls.append({'method': 'schema.relations', 'input': relation_spec})
        return self.inner.relations(relation_spec)

    def __getattr__(self, name: str) -> Any:
        return getattr(self.inner, name)


class RecordingGLiNER2Model:
    def __init__(self, inner: Any) -> None:
        self.inner = inner
        self.schema_calls: list[dict[str, Any]] = []
        self.extract_calls: list[dict[str, Any]] = []

    def create_schema(self) -> RecordingSchema:
        raw = self.inner.create_schema()
        self.schema_calls.append({'method': 'model.create_schema', 'output_type': type(raw).__name__})
        return RecordingSchema(raw, self.schema_calls)

    def extract(self, text: str, schema: Any, threshold: float, *args: Any, **kwargs: Any) -> Any:
        t0 = time.monotonic()
        raw_schema = schema.inner if isinstance(schema, RecordingSchema) else schema
        output = self.inner.extract(text, raw_schema, threshold, *args, **kwargs)
        self.extract_calls.append({
            'method': 'model.extract',
            'input': {'text': text, 'threshold': threshold},
            'output_raw': output,
            'elapsed_s': round(time.monotonic() - t0, 3),
        })
        return output

    def __getattr__(self, name: str) -> Any:
        return getattr(self.inner, name)


inner_schema_miner = _SpacySchemaMiner()
recording_schema_miner = RecordingSchemaMiner(inner_schema_miner)
recorded_model_holder: dict[str, Any] = {}

def recording_model_loader() -> Any:
    inner = load_gliner2_model()
    rm = RecordingGLiNER2Model(inner)
    recorded_model_holder['model'] = rm
    return rm

print('Recording wrappers ready.')

Recording wrappers ready.


### 2a. Demo Case — NovaTech Research Funding Chain (8 sessions, ~4 000 chars)

**Question:** *Which private fund manages the endowment that directly finances Dr. Amara Osei's laboratory?*
**Expected answer: Meridian Endowment Capital**

**Three-hop reasoning chain required:**
1. Dr. Amara Osei directs the **Quantum Materials Lab** at NovaTech Research Institute *(Session 1)*
2. The Quantum Materials Lab's operational funding comes from endowment account **NTI-QMAT-7743** *(Session 2 + 5)*
3. NTI-QMAT-7743 is managed externally by **Meridian Endowment Capital** *(Session 2)*

**Red herring:** Vantage Research Partners manages NovaTech's *other* divisions — not the Quantum Materials Lab. An extractor or retriever that conflates the two funds will answer incorrectly.

The text is long enough (~4 000 chars) to produce non-trivial extraction graphs for both GLiNER2 and DSPy.

In [ ]:
DEMO_CASE = {
    'id': 'novatech-funding-001',
    'question': "Which private fund manages the endowment that directly finances Dr. Amara Osei's laboratory?",
    'answer': 'Meridian Endowment Capital',
    'sessions': [
        # Session 1 — institute context and personnel
        {'messages': [
            {'role': 'user', 'content': (
                "Dr. Amara Osei has just been appointed director of the Quantum Materials Lab at "
                "NovaTech Research Institute in Zurich. She previously held a postdoctoral position "
                "at MIT's CSAIL division under Prof. Daniel Huang, where she co-authored the landmark "
                "paper 'Topological Phase Transitions in Cryogenic Superconductors' that appeared in "
                "Nature Physics in 2022. NovaTech's Executive Director, Dr. Henrik Baumann, personally "
                "recruited her after a competitive international search that lasted eight months."
            )},
            {'role': 'assistant', 'content': (
                "Noted. Dr. Amara Osei is now directing the Quantum Materials Lab at NovaTech Research "
                "Institute in Zurich, previously at MIT CSAIL under Prof. Daniel Huang. Recruited by "
                "Executive Director Dr. Henrik Baumann."
            )},
            {'role': 'user', 'content': (
                "The Quantum Materials Lab has three active EU Horizon grants: TOPOQM-2023, "
                "QMAT-EDGE-2024, and a collaborative project with ETH Zürich called CryoSync. "
                "The CryoSync project focuses on cryogenic superconductor engineering and is "
                "co-led by Prof. Anna Köhler from ETH Zürich's Department of Materials Science. "
                "The project has a three-year budget of €4.2 million, split equally between NovaTech "
                "and ETH Zürich."
            )},
            {'role': 'assistant', 'content': (
                "Noted. Three EU Horizon grants active; CryoSync is the flagship collaboration "
                "with ETH Zürich, co-led by Prof. Anna Köhler, with €4.2M total budget split equally."
            )},
        ]},
        # Session 2 — endowment ledger (critical funding facts)
        {'messages': [
            {'role': 'user', 'content': (
                "NovaTech Research Institute's 2024 endowment ledger shows that the Quantum Materials "
                "Lab receives its primary operational funding — separate from project grants — from "
                "endowment account NTI-QMAT-7743, allocated by the Institute's finance committee "
                "chaired by Dr. Ingrid Larsen, who also represents the Scandinavian Science "
                "Consortium on the NovaTech board."
            )},
            {'role': 'assistant', 'content': (
                "Noted. Quantum Materials Lab core funding: endowment account NTI-QMAT-7743, "
                "finance committee chaired by Dr. Ingrid Larsen."
            )},
            {'role': 'user', 'content': (
                "The endowment account NTI-QMAT-7743 is managed externally by Meridian Endowment "
                "Capital, a private fund headquartered in Geneva. Meridian has been a patron of "
                "NovaTech since 2019 under a perpetual giving agreement signed by Meridian's CEO, "
                "Elara Voss. The agreement commits Meridian to a minimum annual disbursement of "
                "€2.1 million indexed to Swiss CPI inflation."
            )},
            {'role': 'assistant', 'content': (
                "Noted. NTI-QMAT-7743 is externally managed by Meridian Endowment Capital (Geneva), "
                "CEO Elara Voss, patron since 2019 with CPI-indexed €2.1M minimum annual disbursement."
            )},
        ]},
        # Session 3 — conference and publication (contextual entities)
        {'messages': [
            {'role': 'user', 'content': (
                "Dr. Osei presented CryoSync preliminary findings at IEEE Quantum Week 2024 in "
                "Montreal. Her co-presenters were Dr. Raj Patel from ETH Zürich and Dr. Sofia "
                "Mendez from CERN's Applied Physics group. The session was chaired by Prof. "
                "Takeshi Yamamoto from Osaka University, who is also a member of the IEEE "
                "Quantum Initiative steering committee."
            )},
            {'role': 'assistant', 'content': (
                "Noted. IEEE Quantum Week 2024, Montreal. Presenters: Dr. Osei, Dr. Raj Patel "
                "(ETH Zürich), Dr. Sofia Mendez (CERN). Session chair: Prof. Takeshi Yamamoto "
                "(Osaka University)."
            )},
            {'role': 'user', 'content': (
                "The CryoSync paper 'Cryogenic Superconductor Engineering at Scale' has been "
                "submitted to Nature Physics for peer review. The corresponding author is "
                "Dr. Amara Osei. ETH Zürich's portion of CryoSync is funded by the Swiss "
                "National Science Foundation under grant SNSF-CryoSync-2024, while NovaTech's "
                "share is drawn entirely from endowment account NTI-QMAT-7743."
            )},
            {'role': 'assistant', 'content': (
                "Noted. Paper submitted to Nature Physics. ETH funded by SNSF-CryoSync-2024; "
                "NovaTech's share funded from NTI-QMAT-7743."
            )},
        ]},
        # Session 4 — institute governance
        {'messages': [
            {'role': 'user', 'content': (
                "NovaTech Research Institute is governed by a nine-member board. The chair is "
                "Prof. Marcus Weiß, formerly Secretary-General of the Max Planck Society. "
                "The vice-chair is Dr. Ingrid Larsen, who represents the Scandinavian Science "
                "Consortium. Two seats are held by industry representatives from Quantum "
                "Computing Systems AG and Rhenium Advanced Materials GmbH."
            )},
            {'role': 'assistant', 'content': (
                "Noted. Board: Chair Prof. Marcus Weiß (ex-Max Planck), Vice-chair Dr. Ingrid "
                "Larsen (Scandinavian Science Consortium), plus industry seats held by QCS AG "
                "and Rhenium Advanced Materials GmbH."
            )},
            {'role': 'user', 'content': (
                "The institute has five research divisions: Quantum Materials, Bioelectronics, "
                "Climate Modelling, Neuromorphic Computing, and Advanced Robotics. Each division "
                "director reports directly to Executive Director Dr. Henrik Baumann. The division "
                "directors are: Dr. Amara Osei (Quantum Materials), Dr. Fatima Al-Rashid "
                "(Bioelectronics), Dr. Carlos Mendes (Climate Modelling), Dr. Yuki Tanaka "
                "(Neuromorphic Computing), and Dr. Lars Eriksson (Advanced Robotics)."
            )},
            {'role': 'assistant', 'content': (
                "Noted. Five divisions, directors: Osei, Al-Rashid, Mendes, Tanaka, Eriksson, "
                "all reporting to Executive Director Dr. Henrik Baumann."
            )},
        ]},
        # Session 5 — collaboration funding cross-reference
        {'messages': [
            {'role': 'user', 'content': (
                "A budget reconciliation memo from NovaTech's finance office confirms that "
                "Dr. Osei's laboratory operating expenses — including staff salaries for her "
                "six postdoctoral researchers, equipment leases, and consumables — are charged "
                "exclusively to endowment account NTI-QMAT-7743. No EU Horizon grant funds "
                "flow into the lab's operating budget; those grants cover only project-specific "
                "expenditures such as the CryoSync measurement campaign."
            )},
            {'role': 'assistant', 'content': (
                "Noted. All lab operating expenses (salaries, equipment, consumables) charged to "
                "NTI-QMAT-7743. EU Horizon grants cover project expenditures only."
            )},
            {'role': 'user', 'content': (
                "Dr. Osei's postdoctoral researcher Dr. Julian Kang has been offered a faculty "
                "position at Caltech's Department of Applied Physics starting September 2025. "
                "His replacement will be recruited from NovaTech's internal PhD programme. "
                "Dr. Kang's current research on topological qubits at millikelvin temperatures "
                "will be continued by Dr. Osei herself until a successor is hired."
            )},
            {'role': 'assistant', 'content': (
                "Noted. Dr. Julian Kang moving to Caltech Applied Physics in September 2025. "
                "Dr. Osei will cover his topological qubit research until a replacement is found."
            )},
        ]},
        # Session 6 — industry partnership
        {'messages': [
            {'role': 'user', 'content': (
                "Quantum Computing Systems AG, headquartered in Basel, has entered a technology "
                "transfer agreement with NovaTech Research Institute for commercialising findings "
                "from the Quantum Materials Lab. The agreement was negotiated over 14 months by "
                "QCS AG's business development director, Marco Ferreira, and NovaTech's legal "
                "counsel, Dr. Petra Zimmermann. It was signed at a ceremony in Zurich attended "
                "by Dr. Henrik Baumann and QCS AG's CEO, Thomas Brandt."
            )},
            {'role': 'assistant', 'content': (
                "Noted. QCS AG (Basel) tech transfer agreement with NovaTech. Negotiated by "
                "Marco Ferreira (QCS) and Dr. Petra Zimmermann (NovaTech). Signed by Dr. Baumann "
                "and CEO Thomas Brandt."
            )},
            {'role': 'user', 'content': (
                "Under the agreement, QCS AG will provide €800,000 in annual research support "
                "to Dr. Osei's laboratory in exchange for exclusive licensing rights to any "
                "patentable discoveries made using CryoSync infrastructure during the 2024–2026 "
                "period. This QCS AG support is tracked in a separate budget line and does not "
                "affect the endowment account NTI-QMAT-7743 managed by Meridian Endowment Capital."
            )},
            {'role': 'assistant', 'content': (
                "Noted. QCS AG provides €800K/year for exclusive licensing 2024–2026. This is "
                "separate from endowment NTI-QMAT-7743 managed by Meridian Endowment Capital."
            )},
        ]},
        # Session 7 — Dr. Osei's activities
        {'messages': [
            {'role': 'user', 'content': (
                "Dr. Amara Osei will deliver the keynote address at the European Superconductivity "
                "Summit in Vienna in March 2025. She will also serve on the prize committee for the "
                "Von Humboldt Award for Materials Innovation, alongside Prof. Marcus Weiß and "
                "Prof. Takeshi Yamamoto. The award ceremony is scheduled for June 2025 in Berlin."
            )},
            {'role': 'assistant', 'content': (
                "Noted. Dr. Osei: keynote at European Superconductivity Summit (Vienna, March 2025); "
                "Von Humboldt Award committee with Prof. Weiß and Prof. Yamamoto (Berlin, June 2025)."
            )},
            {'role': 'user', 'content': (
                "The Max Planck Society has expressed interest in establishing a joint research "
                "unit with NovaTech's Quantum Materials Lab, building on the existing relationship "
                "through board chair Prof. Marcus Weiß. Preliminary discussions were held between "
                "Dr. Henrik Baumann and the Max Planck President at the Lindau Nobel Laureate "
                "Meetings in July 2024."
            )},
            {'role': 'assistant', 'content': (
                "Noted. Max Planck Society in talks with NovaTech for a joint Quantum Materials "
                "unit. Baumann and Max Planck President discussed at Lindau Meetings, July 2024."
            )},
        ]},
        # Session 8 — red herring: Vantage Research Partners manages OTHER divisions
        {'messages': [
            {'role': 'user', 'content': (
                "The Vantage Research Partners fund, which manages endowments for NovaTech's "
                "Bioelectronics, Climate Modelling, and Neuromorphic Computing divisions, has "
                "recently increased its total allocation to the institute by 12% following a "
                "successful fundraising campaign. The climate modelling division's endowment "
                "account NTI-CLIM-3391 and the bioelectronics account NTI-BIO-2205 are both "
                "managed by Vantage Research Partners, not by Meridian Endowment Capital."
            )},
            {'role': 'assistant', 'content': (
                "Noted. Vantage Research Partners manages NTI-CLIM-3391 (Climate), NTI-BIO-2205 "
                "(Bioelectronics), and Neuromorphic Computing endowments — separate from Meridian "
                "Endowment Capital, which exclusively manages the Quantum Materials Lab account "
                "NTI-QMAT-7743."
            )},
            {'role': 'user', 'content': (
                "To be explicit: endowment account NTI-QMAT-7743, which funds Dr. Amara Osei's "
                "Quantum Materials Lab, is solely managed by Meridian Endowment Capital (CEO: "
                "Elara Voss, Geneva). Vantage Research Partners has no involvement with the "
                "Quantum Materials Lab whatsoever. The two funds operate under separate agreements "
                "with NovaTech and have different investment mandates."
            )},
            {'role': 'assistant', 'content': (
                "Confirmed. NTI-QMAT-7743 → Meridian Endowment Capital (Elara Voss). "
                "Vantage Research Partners manages all other division endowments. Completely separate."
            )},
        ]},
    ],
}

# Flatten all sessions to one trace text (same as eval harness serialization)
def _serialize_session(sess: dict) -> str:
    return '\n'.join(
        f"{m['role']}: {m['content']}"
        for m in sess.get('messages', [])
        if m.get('content', '').strip()
    )

trace_input_text = '\n\n'.join(_serialize_session(s) for s in DEMO_CASE['sessions'])

print(f"Demo case : {DEMO_CASE['id']}")
print(f"Sessions  : {len(DEMO_CASE['sessions'])}")
print(f"Total chars: {len(trace_input_text)}")
print(f"Question  : {DEMO_CASE['question']}")
print(f"Answer    : {DEMO_CASE['answer']}")
print()
print(trace_input_text)

### 2b. Run the LLM-Free Pipeline (GLiNER2 + spaCy)

In [ ]:
extractor = GLiNER2EntityExtractor(
    model_loader=recording_model_loader,
    schema_miner=recording_schema_miner,
)

t0 = time.monotonic()
extraction_result = extractor.extract(trace_input_text)
extract_elapsed = round(time.monotonic() - t0, 3)
recording_model = recorded_model_holder['model']

pipeline_trace = {
    'component': 'GLiNER2EntityExtractor.extract',
    'case': DEMO_CASE['id'],
    'question': DEMO_CASE['question'],
    'expected_answer': DEMO_CASE['answer'],
    'output': {
        'entities': extraction_result.entities,
        'relations': extraction_result.relations,
    },
    'entity_count': len(extraction_result.entities),
    'relation_count': len(extraction_result.relations),
    'elapsed_s': extract_elapsed,
}
trace['components']['pipeline_extract'] = pipeline_trace
show(pipeline_trace)

### 2c. spaCy Schema Hints

spaCy reads the text first and produces a list of candidate entity types and relation patterns. These hints narrow the search space for GLiNER2, improving both speed and precision.

In [ ]:
spacy_hint_trace = recording_schema_miner.calls[-1]
trace['components']['spacy_schema_hints'] = spacy_hint_trace
show(spacy_hint_trace)

<IPython.core.display.JSON object>

### 2d. GLiNER2 Schema Construction

GLiNER2 turns the spaCy hints into a structured schema — a typed list of entity labels and relation patterns — before running its neural extraction pass.

In [ ]:
gliner_schema_trace = {
    'component': 'GLiNER2 schema construction inside _build_gliner_schema',
    'calls': recording_model.schema_calls,
}
trace['components']['gliner_schema_calls'] = gliner_schema_trace
show(gliner_schema_trace)

<IPython.core.display.JSON object>

### 2e. Raw GLiNER2 Output (before post-processing)

In [ ]:
raw_gliner_trace = recording_model.extract_calls[-1]
trace['components']['raw_gliner2_output'] = raw_gliner_trace
show(raw_gliner_trace)

<IPython.core.display.JSON object>

### 2f. Full Pipeline Summary

Side-by-side view of every stage: spaCy hints → GLiNER2 schema calls → raw model output → normalized entities and relations written to the graph.

In [ ]:
debug_log = {
    '1_spacy_hints':        spacy_hint_trace['output'],
    '2_gliner2_schema_calls': recording_model.schema_calls,
    '3_raw_gliner2_output': raw_gliner_trace['output_raw'],
    '4_normalized_result':  pipeline_trace['output'],
}
trace['components']['debug_log'] = debug_log
show(debug_log)

<IPython.core.display.JSON object>

### 2g. GLiNER2 Entity-Relation Graph

Interactive graph of everything the **LLM-free pipeline** found. Node colour encodes entity type (blue = Person, orange = Org/Fund, green = Place, grey = generic). Edge labels show the exact relation type. Edge thickness reflects GLiNER2's confidence score. Click any node to see its type and confidence in the sidebar.

**What to look for:**
- Are `Dr. Amara Osei`, `Quantum Materials Lab`, `NTI-QMAT-7743`, and `Meridian Endowment Capital` all present as nodes?
- Is there a path connecting them with typed edges? If the `manages` or `funds` edge between the endowment account and the fund is missing, GLiNER2 cannot close the three-hop chain.
- Compare node and edge count here vs the DSPy graph in Section 4 — the LLM typically finds more implicit relations.

In [ ]:
# Build yfiles nodes and edges from GLiNER2 extraction result
_g2_seen: set[str] = set()
_g2_nodes: list[dict] = []
_g2_edges: list[dict] = []

for ent in extraction_result.entities:
    nid = ent['name']
    if nid and nid not in _g2_seen:
        _g2_seen.add(nid)
        _g2_nodes.append({
            'id':    nid,
            'label': nid,
            'type':  ent.get('type', 'ENTITY'),
            'properties': {
                'type':       ent.get('type', 'ENTITY'),
                'confidence': round(float(ent.get('score', 0)), 3),
                'extractor':  'GLiNER2',
            },
        })

for i, rel in enumerate(extraction_result.relations):
    src, tgt = rel['head'], rel['tail']
    # Ensure endpoints exist as nodes even if they didn't appear in the entity list
    for nid, ntype in [(src, rel.get('head_type', 'ENTITY')), (tgt, rel.get('tail_type', 'ENTITY'))]:
        if nid and nid not in _g2_seen:
            _g2_seen.add(nid)
            _g2_nodes.append({
                'id': nid, 'label': nid, 'type': ntype,
                'properties': {'type': ntype, 'extractor': 'GLiNER2'},
            })
    if src and tgt:
        _g2_edges.append({
            'id':    f'g2-{i}',
            'start': src,
            'end':   tgt,
            'label': rel.get('relation', '?'),
            'properties': {
                'relation':   rel.get('relation', '?'),
                'head_type':  rel.get('head_type', ''),
                'tail_type':  rel.get('tail_type', ''),
                'confidence': round(float(rel.get('score', 0)), 3),
                'extractor':  'GLiNER2',
            },
        })

_plot_graph(
    _g2_nodes, _g2_edges,
    title=f'Pipeline A — GLiNER2 LLM-free extraction  ·  {DEMO_CASE["id"]}',
    layout='organic',
    height='680px',
)

## Section 3 — Ontology Router

Different content types call for different vocabularies. A medical document needs `Patient`, `Diagnosis`, `Treatment`; a conversation about companies needs `Person`, `Organization`, `MemberOf`.

`OntologyRouter` reads the text and scores it against every registered ontology, then picks the best fit. For the locomo-006 case (people, foundations, financial transfers) it should select **schema-org** — the broad cross-domain vocabulary covering Person, Organization, Event, and CreativeWork.

In [ ]:
loader = OntologyLoader()
router = OntologyRouter(loader=loader)

# Route using the combined locomo-006 text
selected = router.select(trace_input_text)
top3 = router.select_multi(trace_input_text, top_k=3)

ontology_trace = {
    'selected': {
        'id': selected.id,
        'name': selected.name,
        'description': selected.description,
        'domains': selected.domains,
        'class_count': len(selected.class_labels),
        'property_count': len(selected.property_labels),
        'class_labels': selected.class_labels,
        'property_labels': selected.property_labels,
    },
    'ranking': [
        {'id': m.id, 'name': m.name, 'class_count': len(m.class_labels)}
        for m in top3
    ],
}
trace['components']['ontology_routing'] = ontology_trace
show(ontology_trace)

<IPython.core.display.JSON object>

## Section 4 — Pipeline B: LLM-Assisted Extraction (DSPy + OpenRouter)

The same NovaTech conversation text, now processed by `DSPyExtractor`. Instead of a DeBERTa model with spaCy hints, the LLM receives the full text and an open-domain vocabulary of **12 entity classes and 39 relation types** aligned with schema-org, then reasons about what relationships exist.

**Why this matters:** GLiNER2 excels at finding entities that are *explicitly named* in text and their direct syntactic relations. The LLM can additionally find:
- **Implicit relations** — "Elara Voss leads Meridian Endowment Capital" inferred from "Meridian's CEO, Elara Voss"
- **Multi-sentence links** — "NTI-QMAT-7743 is managed by Meridian" + "Meridian funds Dr. Osei's lab" → `manages_funding_of`
- **Typed attribution** — correctly classifying NTI-QMAT-7743 as a financial instrument rather than a generic entity

The graph that follows should be **visibly denser** than the GLiNER2 graph: more edges, richer relation labels, and better entity typing.

Requires `OPENROUTER_API_KEY` in `.env`.

In [ ]:
# Schema-org loaded via OntologyLoader (for display — shows alignment with open-domain vocab)
schema_org_manifest = loader.load('schema-org')

dspy_trace: dict[str, Any] = {
    'enabled': bool(RUN_DSPY_LLM_ASSISTED),
    'case': DEMO_CASE['id'],
    'ontology_mode': 'open-domain (no RDF file — vocabulary matches schema-org semantics)',
    'open_domain_classes': _OPEN_DOMAIN_CLASSES,
    'open_domain_relation_types': _OPEN_DOMAIN_RELATION_TYPES,
    'schema_org_reference': {
        'id': schema_org_manifest.id,
        'class_labels': schema_org_manifest.class_labels,
        'property_labels': schema_org_manifest.property_labels,
    },
}

if not RUN_DSPY_LLM_ASSISTED:
    dspy_trace['status'] = 'skipped: RUN_DSPY_LLM_ASSISTED=False'
elif not (settings.openrouter_api_key or '').strip():
    dspy_trace['status'] = 'skipped: OPENROUTER_API_KEY not configured'
else:
    dspy_extractor = DSPyExtractor(
        config={
            'llm': {'model': LLM_MODEL},
            'infra': {'openrouter_api_key': settings.openrouter_api_key or ''},
            'ontology': {'file': ''},  # open-domain mode
        }
    )
    t0 = time.monotonic()
    dspy_relations, dspy_nodes, dspy_audit = await dspy_extractor.extract_relations(text=trace_input_text)
    dspy_trace.update({
        'status': 'ran',
        'elapsed_s': round(time.monotonic() - t0, 3),
        'raw_triplets': dspy_audit.get('raw_triplets', []),
        'triplet_decisions': dspy_audit.get('triplet_decisions', []),
        'relations_for_graph_generation': [
            {'source': r[0], 'relation': r[1], 'target': r[2], 'props': r[3]}
            for r in dspy_relations
        ],
        'nodes_for_graph_generation': dspy_nodes,
        'relation_count': len(dspy_relations),
    })

trace['components']['dspy_extraction'] = dspy_trace
show(dspy_trace)

### 4b. DSPy LLM-Assisted Relation Graph

The same NovaTech text, now extracted by **DSPy + OpenRouter LLM**. The model uses 12 open-domain entity classes and 39 relation types aligned with schema-org. Compare this with the GLiNER2 graph in Section 2 to see what the LLM adds.

**What to look for:**
- Does the LLM correctly surface `Meridian Endowment Capital` as the manager of `NTI-QMAT-7743`?
- Are implicit role edges present — e.g. `Elara Voss → CEO_OF → Meridian Endowment Capital` inferred from appositive phrasing?
- Is `Vantage Research Partners` correctly scoped to the *other* division accounts (`NTI-CLIM-3391`, `NTI-BIO-2205`) and kept separate from the Quantum Materials Lab?
- Richer entity typing (ORG, PERSON) and a denser graph are expected; any edge that closes the three-hop chain is what LTM graph traversal exploits in Section 6.

In [ ]:
if dspy_trace.get('status') != 'ran':
    print(f'DSPy extraction was skipped ({dspy_trace.get("status", "no status")}) — no graph to render.')
else:
    _dp_nodes_raw = dspy_trace.get('nodes_for_graph_generation', [])
    _dp_rels_raw  = dspy_trace.get('relations_for_graph_generation', [])

    _dp_seen: set[str] = set()
    _dp_nodes: list[dict] = []
    _dp_edges: list[dict] = []

    for n in _dp_nodes_raw:
        nid = str(n.get('id', ''))
        if nid and nid not in _dp_seen:
            _dp_seen.add(nid)
            _dp_nodes.append({
                'id': nid,
                'label': nid,
                'type': n.get('type', 'ENTITY'),
                'properties': {
                    'type':      n.get('type', 'ENTITY'),
                    'extractor': 'DSPy',
                    **(n.get('properties') or {}),
                },
            })

    for i, rel in enumerate(_dp_rels_raw):
        src, tgt = rel.get('source', ''), rel.get('target', '')
        rel_props = {k: v for k, v in (rel.get('props') or {}).items()}
        rel_props['extractor'] = 'DSPy'
        for nid in (src, tgt):
            if nid and nid not in _dp_seen:
                _dp_seen.add(nid)
                _dp_nodes.append({
                    'id': nid, 'label': nid, 'type': 'ENTITY',
                    'properties': {'type': 'ENTITY', 'extractor': 'DSPy'},
                })
        if src and tgt:
            _dp_edges.append({
                'id': f'dp-{i}',
                'start': src,
                'end': tgt,
                'label': rel.get('relation', '?'),
                'properties': rel_props,
            })

    _plot_graph(
        _dp_nodes, _dp_edges,
        title=f'Pipeline B — DSPy LLM-assisted extraction  ·  {DEMO_CASE["id"]}',
        layout='organic',
        height='680px',
    )

### 4c. Pipeline A vs Pipeline B — Side-by-Side Comparison

Both extractors ran on the same 8-session NovaTech text (`novatech-funding-001`). The key question: does the LLM find the `Meridian Endowment Capital → manages → NTI-QMAT-7743` link and correctly ignore the red-herring `Vantage Research Partners` fund — where GLiNER2 might conflate the two?

In [ ]:
relation_comparison = {
    'case': DEMO_CASE['id'],
    'question': DEMO_CASE['question'],
    'expected_answer': DEMO_CASE['answer'],
    'llm_free_gliner2': {
        'entities': extraction_result.entities,
        'relations': extraction_result.relations,
        'entity_count': len(extraction_result.entities),
        'relation_count': len(extraction_result.relations),
    },
    'dspy_llm_assisted': {
        'status': dspy_trace.get('status', 'ran'),
        'relations': dspy_trace.get('relations_for_graph_generation', []),
        'relation_count': dspy_trace.get('relation_count', 0),
    },
}
trace['components']['relation_comparison'] = relation_comparison
show(relation_comparison)

## Section 5 — Short-Term Memory (STM): Ingest + Recall

Short-term memory holds the raw conversation and document chunks in **Memgraph** (for graph queries) and **pgvector** (for semantic search). It's fast to write and query, but limited to a single session — there's no cross-session entity linking yet.

Each of the 6 cases is ingested, then recalled against its own question. The **answer-support score** is 1.0 if the expected answer text appears in the top-5 retrieved chunks, 0.0 if it doesn't. Multi-hop LoCoMo cases are expected to score low here because the answer requires connecting facts across sessions — that's what LTM is for.

In [ ]:
def _answer_support_score(answer: Any, recall_results: list[Any]) -> float:
    """Same metric as the eval harness: substring match score over recall hits."""
    expected = str(answer or '').strip().lower()
    if not expected:
        return 0.0
    haystack = ' '.join(getattr(r, 'text', str(r)) for r in recall_results).lower()
    if expected in haystack:
        return 1.0
    tokens = set(re.findall(r'[a-z0-9][a-z0-9_-]{1,}', expected))
    if not tokens:
        return 0.0
    overlap = tokens & set(re.findall(r'[a-z0-9][a-z0-9_-]{1,}', haystack))
    return round(len(overlap) / len(tokens), 3)


stm_results: list[dict[str, Any]] = []

if not RUN_STM_LTM_WRITEBACK:
    print('Skipped: RUN_STM_LTM_WRITEBACK=False')
else:
    stm_service = build_stm_service(settings)

    for qid in LOCOMO_CASE_IDS + BEAM_CASE_IDS:
        case = locomo_cases.get(qid) or beam_cases.get(qid)
        bm = 'locomo' if qid.startswith('locomo') else 'beam'
        session_id = f'{run_id}-{qid}'
        docs = _case_documents(case)

        # Ingest all source documents
        ingest_stats: list[dict] = []
        t0 = time.monotonic()
        for doc in docs:
            res = stm_service.ingest(
                text=doc['text'],
                session_id=session_id,
                title=doc['title'],
                source_hint=f'eval/fixtures/{bm}/questions.jsonl',
                metadata={'case_id': qid, 'benchmark': bm},
            )
            ingest_stats.append({'title': doc['title'], 'chunks': res.chunk_count, 'entities': res.entity_count})
        ingest_elapsed = round(time.monotonic() - t0, 3)

        # STM recall
        t1 = time.monotonic()
        recall_hits = stm_service.recall(case['question'], session_id=session_id, top_k=5)
        recall_elapsed = round(time.monotonic() - t1, 3)

        score = _answer_support_score(case['answer'], recall_hits)

        result = {
            'case_id': qid,
            'benchmark': bm,
            'category': case.get('category', case.get('question_type', '?')),
            'question': case['question'],
            'expected_answer': str(case['answer']),
            'session_id': session_id,
            'ingest': {
                'documents_ingested': len(docs),
                'per_document': ingest_stats,
                'total_elapsed_s': ingest_elapsed,
            },
            'stm_recall': {
                'top_k': 5,
                'hits_returned': len(recall_hits),
                'elapsed_s': recall_elapsed,
                'top_hits': [
                    {
                        'score': round(h.score, 4),
                        'source_type': h.source_type,
                        'entity_names': h.entity_names[:6],
                        'text_preview': h.text[:160],
                    }
                    for h in recall_hits[:3]
                ],
            },
            'answer_support_score': score,
            'answer_support_hit': score >= 1.0,
        }
        stm_results.append(result)

        icon = '✓' if score >= 1.0 else ('~' if score > 0 else '✗')
        print(f'{icon} [{bm}] {qid} | score={score:.3f} | Q: {case["question"][:60]}')

trace['components']['stm_results'] = stm_results

✗ [locomo] locomo-smoke-006 | score=0.000 | Q: Which company is the patron of the institute Elena recently 


✗ [locomo] locomo-smoke-007 | score=0.000 | Q: What is the transaction code for the proposal Carlos recentl


✗ [locomo] locomo-smoke-008 | score=0.000 | Q: What is the badge number of the person Kenji reports to?


✓ [beam] beam-10m-smoke-001 | score=1.000 | Q: Which warehouse should handle the Vega recall after the fina


✓ [beam] beam-10m-smoke-003 | score=1.000 | Q: Which event happened first: the server migration or the fire


✓ [beam] beam-10m-smoke-005 | score=1.000 | Q: Which team owns the service that stores customer payment rec


### 5a. LoCoMo Cases — Detailed Results

These three cases require multi-session memory. STM recall scores are expected to be low because the answer is never stated directly in a single session.

In [ ]:
locomo_results = [r for r in stm_results if r['benchmark'] == 'locomo']
show(locomo_results)

<IPython.core.display.JSON object>

### 5b. BEAM Cases — Detailed Results

BEAM cases present the answer in one document among several that sometimes conflict. STM recall tends to score higher here because the answer text is present verbatim in the ingested content.

In [ ]:
beam_results = [r for r in stm_results if r['benchmark'] == 'beam']
show(beam_results)

<IPython.core.display.JSON object>

### 5c. Score Summary

A score of 1.0 means the expected answer was found in the top-5 retrieved chunks. Scores below 1.0 indicate partial overlap.

**LoCoMo cases score 0.0 here — this is by design, not a bug.** Each LoCoMo answer requires linking facts that are spread across multiple separate conversation sessions. A vector search over single-session chunks cannot make that cross-session connection; only an entity graph can. Section 6 shows how LTM promotion + graph traversal recovers these answers.

In [ ]:
summary_table = [
    {
        'case_id': r['case_id'],
        'benchmark': r['benchmark'],
        'category': r['category'],
        'question': r['question'],
        'expected': r['expected_answer'],
        'stm_score': r['answer_support_score'],
        'stm_hit': r['answer_support_hit'],
    }
    for r in stm_results
]
show(summary_table)

hits = sum(1 for r in stm_results if r['answer_support_hit'])
avg  = round(sum(r['answer_support_score'] for r in stm_results) / max(len(stm_results), 1), 3)
print(f'\nSTM recall:  {hits}/{len(stm_results)} exact hits  |  avg_score={avg}')

<IPython.core.display.JSON object>


STM recall:  3/6 exact hits  |  avg_score=0.5


## Section 6 — Long-Term Memory (LTM): Promotion + DETRetriever

**Why this section matters:** Section 5 showed STM scoring 0/3 on LoCoMo questions — the answer text was in the corpus, but vector search alone can't cross session boundaries to link facts. LTM graph promotion fixes this by merging entities across sessions and building a traversable knowledge graph.

**Promotion** moves memories from the per-session short-term store into a shared long-term knowledge graph in Neo4j. During this process, entities are merged across sessions, community detection (Leiden algorithm) groups related entities into topics, and PageRank scores nodes by their importance.

After promotion, the **DETRetriever** (Deterministic Entity-Traversal) runs four complementary search strategies in parallel, then merges all results using Reciprocal Rank Fusion (RRF) — chunks that are found by multiple strategies rank higher than chunks found by only one:

| Strategy | Where | What it finds |
|---|---|---|
| Entity boost | Memgraph | Chunks that mention entities from the query |
| Topic recall | Neo4j | LTM topic nodes whose entities match query keywords |
| Entity traversal | Neo4j | Chunks reachable by following entity→relation→entity links |
| Semantic ANN | pgvector | Chunks whose embedding is closest to the query embedding |

The demo case is `locomo-smoke-006` — the question whose answer requires connecting three facts across different sessions:
1. Elena joined the Nexus Foundation (session 1)
2. MCF-7734 was disbursed by Meridian Capital to Nexus Foundation (session 2)
3. Therefore Meridian Capital is the patron → this link only exists in the graph

In [ ]:
ltm_trace: dict[str, Any] = {'enabled': RUN_STM_LTM_WRITEBACK, 'case_id': LTM_DEMO_CASE_ID}
writeback_flusher = None

# Find session_id for the demo case
ltm_session_id = f'{run_id}-{LTM_DEMO_CASE_ID}'
ltm_case = locomo_cases.get(LTM_DEMO_CASE_ID) or beam_cases.get(LTM_DEMO_CASE_ID)

if RUN_STM_LTM_WRITEBACK:
    flusher = build_promotion_pipeline(settings)
    flusher._llm_model = LLM_MODEL

    t0 = time.monotonic()
    flush_result = await flusher.promote(
        PromotionRequest(
            session_id=ltm_session_id,
            clear_stm=False,
            run_gds=RUN_GDS,
            run_topics=True,
            merge_entities=True,
        )
    )
    ltm_trace.update({
        'session_id': ltm_session_id,
        'flush_result': flush_result.model_dump(),
        'elapsed_s': round(time.monotonic() - t0, 3),
    })
    writeback_flusher = flusher
else:
    ltm_trace['status'] = 'skipped: RUN_STM_LTM_WRITEBACK=False'

trace['components']['ltm_promotion'] = ltm_trace
show(ltm_trace)

<IPython.core.display.JSON object>

### 6a. DETRetriever — Unified Hybrid Search

A single `retrieve()` call runs all four strategies concurrently and returns a ranked, fused result list. The `sources` field on each result shows which strategies contributed to it — multi-source hits rank higher.

**What to look for:** If LTM graph traversal is working, `Meridian Capital` should appear in the results even though no single session chunk directly answers "which company is the patron." The entity graph links `Elena → Nexus Foundation → Meridian Capital`.

In [ ]:
retrieval_question = ltm_case['question'] if ltm_case else 'Who manages the organization?'
retrieval_trace: dict[str, Any] = {'enabled': RUN_STM_LTM_WRITEBACK, 'query': retrieval_question}

if not RUN_STM_LTM_WRITEBACK:
    retrieval_trace['status'] = 'skipped: RUN_STM_LTM_WRITEBACK=False'
    print('Skipped.')
else:
    async with GraphKnowsContainer.from_settings(settings) as gk:
        retriever: DETRetriever = gk.retriever  # type: ignore[assignment]

        ctx = await retriever.retrieve(retrieval_question, session_id=ltm_session_id, top_k=5)

        # Compute answer support score — does the expected answer appear in retrieved text?
        answer_score = _answer_support_score(
            ltm_case['answer'],
            [type('Hit', (), {'text': r['text']})() for r in ctx.fused_results],
        )
        answer_found = answer_score >= 1.0

        # ------------------------------------------------------------------
        # Summary — the key proof point for LTM graph traversal
        # ------------------------------------------------------------------
        stm_score_for_case = next(
            (r['answer_support_score'] for r in stm_results if r['case_id'] == LTM_DEMO_CASE_ID),
            None,
        )
        print('=' * 60)
        print(f'Question : {retrieval_question}')
        print(f'Expected : {ltm_case["answer"]}')
        print()
        if stm_score_for_case is not None:
            stm_verdict = '✓' if stm_score_for_case >= 1.0 else '✗'
            print(f'  STM recall score  : {stm_verdict}  {stm_score_for_case:.3f}  (vector search only)')
        ltm_verdict = '✓ FOUND' if answer_found else '✗ NOT FOUND'
        print(f'  LTM graph recall  : {ltm_verdict}   score={answer_score:.3f}  (graph traversal + vector)')
        print('=' * 60)
        print()
        print(f'Results returned: {len(ctx.fused_results)}')
        print()
        for i, r in enumerate(ctx.fused_results, 1):
            # sources is a comma-joined string like "ltm_graph,semantic"
            sources = r.get('sources') or 'unknown'
            print(f'  #{i}  RRF score={r["rrf_score"]:.4f}  sources=[{sources}]')
            print(f'       {r["text"][:160]}')
            print()

        det_out = {
            'search_type': ctx.search_type,
            'query': retrieval_question,
            'expected_answer': str(ltm_case['answer']),
            'stm_score': stm_score_for_case,
            'ltm_answer_support_score': answer_score,
            'ltm_answer_found': answer_found,
            'results_returned': len(ctx.fused_results),
            'fused_results': [
                {
                    'rank': r['rank'],
                    'rrf_score': round(r['rrf_score'], 4),
                    'sources': r.get('sources', ''),
                    'entities': r.get('entities', []),
                    'text_preview': r['text'][:200],
                }
                for r in ctx.fused_results
            ],
        }
        show(det_out)
        retrieval_trace.update(det_out)

trace['components']['retrieval'] = retrieval_trace
print('\nRetrieval section complete.')

### 6b. Knowledge Graph View — LTM Entities, Relations & Topic Clusters

The Neo4j knowledge graph that was built by the flush pipeline. Three node types are visible:

| Shape / colour | Meaning |
|---|---|
| **Blue** — Person | Individuals extracted from the conversations |
| **Orange** — Org | Companies, foundations, institutions |
| **Amber (large)** — Topic | Leiden community cluster with a GPT-generated summary |
| **Grey** — Entity | Generic entities that didn't match a richer type |

Edge labels show the extracted relation type. Edge thickness is proportional to the relation weight (how many times this relation was observed). **Click any node** to see all its metadata in the sidebar: type, PageRank score, Leiden community ID, confidence, and — for topics — the full generated summary.

**Four graph views are rendered in sequence:**
1. **Entity graph** — only ENTITY nodes and their typed relations (clean, no topic noise)
2. **Full graph** — entities + topic cluster nodes + `IN_TOPIC` membership edges + `CO_OCCURS` co-occurrence edges

In [ ]:
async def _fetch_ltm_graph(neo: Any) -> dict[str, Any]:
    """Query Neo4j for the full LTM graph with all node and edge metadata."""

    entity_rows = await neo._arun(
        'MATCH (e:ENTITY) '
        'RETURN e.name AS id, e.name AS label, '
        '       coalesce(e.type, "ENTITY") AS type, '
        '       coalesce(e.pagerank, 0.0) AS pagerank, '
        '       coalesce(e.community_id, -1) AS community_id, '
        '       coalesce(e.confidence, 1.0) AS confidence, '
        '       coalesce(e.last_seen, "") AS last_seen '
        'LIMIT 150'
    )
    rel_rows = await neo._arun(
        'MATCH (a:ENTITY)-[r]->(b:ENTITY) '
        'WHERE NOT type(r) IN ["IN_TOPIC", "CO_OCCURS"] '
        'RETURN a.name AS source, b.name AS target, '
        '       type(r) AS label, '
        '       coalesce(r.weight, 1) AS weight '
        'LIMIT 300'
    )
    co_occurs = await neo._arun(
        'MATCH (a:ENTITY)-[r:CO_OCCURS]->(b:ENTITY) '
        'RETURN a.name AS source, b.name AS target, '
        '       coalesce(r.weight, 1) AS weight '
        'LIMIT 200'
    )
    topic_rows = await neo._arun(
        'MATCH (t:TOPIC) '
        'RETURN t.id AS id, '
        '       coalesce(t.label, t.id) AS label, '
        '       coalesce(t.summary, "") AS summary, '
        '       coalesce(t.community_id, -1) AS community_id, '
        '       coalesce(t.community_algorithm, "leiden") AS algorithm '
        'LIMIT 50'
    )
    topic_edges = await neo._arun(
        'MATCH (e:ENTITY)-[:IN_TOPIC]->(t:TOPIC) '
        'RETURN e.name AS source, t.id AS target '
        'LIMIT 300'
    )

    # --- Build node list ---
    entity_nodes: list[dict] = []
    for r in entity_rows:
        rd = dict(r)
        pr = float(rd.get('pagerank') or 0)
        entity_nodes.append({
            'id':    rd['id'],
            'label': rd['label'],
            'type':  rd.get('type') or 'ENTITY',
            'properties': {
                'type':         rd.get('type', 'ENTITY'),
                'pagerank':     round(pr, 4),
                'community_id': rd.get('community_id'),
                'confidence':   round(float(rd.get('confidence') or 1.0), 3),
                'last_seen':    rd.get('last_seen', ''),
            },
        })

    topic_nodes: list[dict] = []
    for r in topic_rows:
        rd = dict(r)
        summary = rd.get('summary') or ''
        topic_nodes.append({
            'id':    rd['id'],
            'label': rd['label'],
            'type':  'TOPIC',
            'properties': {
                'type':         'TOPIC',
                'summary':      (summary[:300] + '…') if len(summary) > 300 else summary,
                'community_id': rd.get('community_id'),
                'algorithm':    rd.get('algorithm', 'leiden'),
            },
        })

    # --- Build edge list ---
    relation_edges: list[dict] = []
    for i, r in enumerate(rel_rows):
        rd = dict(r)
        relation_edges.append({
            'id':    f'rel-{i}',
            'start': rd['source'],
            'end':   rd['target'],
            'label': rd['label'],
            'properties': {'weight': float(rd.get('weight') or 1), 'edge_type': 'RELATION'},
        })
    co_edges: list[dict] = []
    for i, r in enumerate(co_occurs):
        rd = dict(r)
        co_edges.append({
            'id':    f'co-{i}',
            'start': rd['source'],
            'end':   rd['target'],
            'label': 'CO_OCCURS',
            'properties': {'weight': float(rd.get('weight') or 1), 'edge_type': 'CO_OCCURS'},
        })
    membership_edges: list[dict] = []
    for i, r in enumerate(topic_edges):
        rd = dict(r)
        membership_edges.append({
            'id':    f'tp-{i}',
            'start': rd['source'],
            'end':   rd['target'],
            'label': 'IN_TOPIC',
            'properties': {'edge_type': 'IN_TOPIC'},
        })

    return {
        'entity_nodes':      entity_nodes,
        'topic_nodes':       topic_nodes,
        'relation_edges':    relation_edges,
        'co_edges':          co_edges,
        'membership_edges':  membership_edges,
    }


# ---- Render ----------------------------------------------------------------
graph_payload = None

if writeback_flusher is not None:
    _ltm = await _fetch_ltm_graph(writeback_flusher._neo)

    # Flatten for JSON export (backwards-compatible key)
    graph_payload = {
        'nodes': _ltm['entity_nodes'] + _ltm['topic_nodes'],
        'edges': _ltm['relation_edges'] + _ltm['co_edges'] + _ltm['membership_edges'],
    }
    trace['components']['graph_payload'] = graph_payload

    n_ent  = len(_ltm['entity_nodes'])
    n_top  = len(_ltm['topic_nodes'])
    n_rel  = len(_ltm['relation_edges'])
    n_co   = len(_ltm['co_edges'])
    n_mem  = len(_ltm['membership_edges'])
    print(f'LTM graph fetched: {n_ent} entities · {n_top} topics · '
          f'{n_rel} relations · {n_co} co-occurs · {n_mem} IN_TOPIC edges')
    print()

    # --- View 1: Entity-only graph (typed relations, no topics) ---
    _plot_graph(
        _ltm['entity_nodes'],
        _ltm['relation_edges'],
        title='View 1 — Entity relation graph  (no topics)',
        layout='organic',
        height='600px',
    )

    # --- View 2: Full graph including topic clusters and co-occurrence ---
    _plot_graph(
        _ltm['entity_nodes'] + _ltm['topic_nodes'],
        _ltm['relation_edges'] + _ltm['membership_edges'] + _ltm['co_edges'],
        title='View 2 — Full LTM graph  (entities + topics + co-occurrence)',
        layout='organic',
        height='800px',
    )

else:
    print('Skipped — no flusher (set RUN_STM_LTM_WRITEBACK = True to enable).')

## Section 7 — Benchmark Comparison

Loads saved result files from `evaluation/results/` and compares two extraction modes side by side:
- **GLiNER2** (LLM-free, fast, no API key needed)
- **LLM-assisted** (DSPy + OpenRouter, slower, finds deeper relations)

**To generate comparison data,** run the evaluation harness with both modes. Then re-run this section:

```bash
# LLM-free baseline (fast, no API key)
python -m evaluation locomo --promote-to-ltm --recall-scope both --limit 10 \
       --relation-extraction-mode gliner2

# LLM-assisted (needs OPENROUTER_API_KEY in .env)
python -m evaluation locomo --promote-to-ltm --recall-scope both --limit 10 \
       --relation-extraction-mode llm_assisted
```

If `evaluation/results/` is empty, the comparison table below will show a placeholder row.

Only result files tagged with `promote_to_ltm: true` are included — STM-only runs are not comparable.

In [ ]:
SUPPORTED_BENCHMARKS = ('locomo', 'longmemeval', 'beam')
RESULT_ROOT = REPO_ROOT / 'evaluation' / 'results'

def _load_report(path: Path) -> dict[str, Any]:
    r = json.loads(path.read_text(encoding='utf-8'))
    missing = sorted({'benchmark', 'run_id', 'summary', 'cases'} - set(r))
    if missing:
        raise ValueError(f'{path} missing {missing}')
    if not isinstance(r.get('cases'), list):
        raise ValueError(f'{path} has non-list cases')
    return r

def _infer_mode(report: dict, path: Path) -> str | None:
    meta = report.get('metadata', {}) or {}
    direct = meta.get('relation_extraction_mode') or meta.get('relation_mode')
    if direct in {'gliner2', 'llm_assisted', 'off'}:
        return str(direct)
    haystack = f"{report.get('run_id', '')} {path.name}".lower()
    if 'llm_assisted' in haystack: return 'llm_assisted'
    if 'gliner2' in haystack or 'llm_free' in haystack: return 'gliner2'
    if '-off-' in haystack or haystack.endswith('-off'): return 'off'
    return None

def _discover() -> dict[str, dict[str, dict]]:
    bucket_of = {'gliner2': 'llm_free', 'llm_assisted': 'llm_assisted', 'off': 'relation_off'}
    found: dict[str, dict[str, dict]] = {}
    if not RESULT_ROOT.exists():
        return found
    for path in sorted(RESULT_ROOT.glob('**/*.json'), key=lambda p: p.stat().st_mtime, reverse=True):
        try:
            r = _load_report(path)
        except Exception:
            continue
        bm = str(r.get('benchmark', '')).lower()
        if bm not in SUPPORTED_BENCHMARKS:
            continue
        meta = r.get('metadata', {}) or {}
        if meta.get('promote_to_ltm') is not True:
            continue
        bucket = bucket_of.get(_infer_mode(r, path) or '')
        if bucket:
            found.setdefault(bm, {}).setdefault(bucket, {'path': path, 'report': r})
    return found

discovered = _discover()
show({'discovered': {bm: list(modes.keys()) for bm, modes in discovered.items()}})

<IPython.core.display.JSON object>

In [ ]:
def _metrics(report: dict) -> dict[str, float]:
    s = report.get('summary', {}) or {}
    out: dict[str, float] = {}
    if isinstance(s.get('avg_score'), (int, float)):
        out['avg_score'] = float(s['avg_score'])
    for k, v in (s.get('metric_summary', {}) or {}).items():
        if isinstance(v, bool):   out[k] = 1.0 if v else 0.0
        elif isinstance(v, (int, float)): out[k] = float(v)
    return out

def _compare(bm: str, modes: dict) -> dict[str, Any]:
    checks: list[str] = []
    rows:   list[dict] = []
    free, asst = modes.get('llm_free'), modes.get('llm_assisted')
    if not free:  checks.append('missing llm_free/gliner2 report')
    if not asst:  checks.append('missing llm_assisted report')
    if not free or not asst:
        return {'benchmark': bm, 'status': 'incomplete', 'checks': checks, 'metric_rows': rows}
    fm, am = _metrics(free['report']), _metrics(asst['report'])
    for metric in sorted(set(fm) | set(am)):
        fv, av = fm.get(metric), am.get(metric)
        rows.append({'benchmark': bm, 'metric': metric, 'gliner2': fv, 'llm_assisted': av,
                     'delta': round(av - fv, 4) if fv is not None and av is not None else None})
    return {'benchmark': bm, 'status': 'complete', 'checks': checks, 'metric_rows': rows}

comparison = {bm: _compare(bm, discovered.get(bm, {})) for bm in SUPPORTED_BENCHMARKS}
trace['components']['benchmark_comparison'] = comparison

_R = {'avg_score', 'answer_support_hit', 'answer_support_score'}
_G = {'flush_edges', 'flush_nodes', 'flush_chunks', 'flush_topics'}

def _f(v: Any) -> str:
    if v is None: return '-'
    if isinstance(v, float): return (f'+{v:.3f}' if v > 0 else f'{v:.3f}')
    return str(v)

lines = ['## Benchmark Results: gliner2 vs llm_assisted', '',
         '| Benchmark | Metric | gliner2 | llm_assisted | Δ |',
         '| --- | --- | ---: | ---: | ---: |']
any_rows = False
for bm in SUPPORTED_BENCHMARKS:
    for row in comparison[bm].get('metric_rows', []):
        if row['metric'] in _R or row['metric'] in _G:
            lines.append(f"| {bm} | {row['metric']} | {_f(row['gliner2'])} | {_f(row['llm_assisted'])} | {_f(row['delta'])} |")
            any_rows = True
if not any_rows:
    lines.append('| - | No complete report pairs in evaluation/results/ | - | - | - |')

issues = [f"- `{bm}`: {'; '.join(comparison[bm].get('checks', []))}"
          for bm in SUPPORTED_BENCHMARKS if comparison[bm].get('checks')]
if issues:
    lines += ['', '### Notes', ''] + issues

try:
    from IPython.display import Markdown, display
    display(Markdown('\n'.join(lines)))
except Exception:
    print('\n'.join(lines))

## Benchmark Results: gliner2 vs llm_assisted

| Benchmark | Metric | gliner2 | llm_assisted | Δ |
| --- | --- | ---: | ---: | ---: |
| locomo | answer_support_hit | +1.000 | +1.000 | 0.000 |
| locomo | answer_support_score | +1.000 | +1.000 | 0.000 |
| locomo | avg_score | +1.000 | +1.000 | 0.000 |
| locomo | flush_chunks | +1.800 | +1.800 | 0.000 |
| locomo | flush_edges | +8.600 | +12.800 | +4.200 |
| locomo | flush_nodes | +7.200 | +7.200 | 0.000 |
| locomo | flush_topics | 0.000 | 0.000 | 0.000 |
| longmemeval | answer_support_hit | +1.000 | +1.000 | 0.000 |
| longmemeval | answer_support_score | +1.000 | +1.000 | 0.000 |
| longmemeval | avg_score | +1.000 | +1.000 | 0.000 |
| longmemeval | flush_chunks | +1.000 | +1.000 | 0.000 |
| longmemeval | flush_edges | +5.000 | +7.000 | +2.000 |
| longmemeval | flush_nodes | +5.000 | +5.000 | 0.000 |
| longmemeval | flush_topics | 0.000 | 0.000 | 0.000 |
| beam | answer_support_hit | +1.000 | +1.000 | 0.000 |
| beam | answer_support_score | +1.000 | +1.000 | 0.000 |
| beam | avg_score | +1.000 | +1.000 | 0.000 |
| beam | flush_chunks | +2.000 | +2.000 | 0.000 |
| beam | flush_edges | +6.600 | +10.800 | +4.200 |
| beam | flush_nodes | +6.400 | +6.400 | 0.000 |
| beam | flush_topics | 0.000 | 0.000 | 0.000 |

## Section 8 — Memory Stats After Demo

Shows how many nodes, chunks, and turns were written to each backend during this run. Compare the before/after counts to verify the ingest and promotion steps worked.

In [ ]:
memory_stats_after = await get_memory_stats(settings)
trace['memory_stats_after'] = memory_stats_after

show({
    'before': trace['memory_stats_before'],
    'after':  memory_stats_after,
})

<IPython.core.display.JSON object>

## Section 9 — Save Trace

Writes the full run trace to a JSON file under `research/notebook_artifacts/<run_id>/`. The trace contains every input, output, and timing measurement from all previous sections — useful for debugging or comparing runs.

In [ ]:
trace_path = artifact_dir / 'demo_trace.json'
trace_path.write_text(json.dumps(trace, indent=2, ensure_ascii=False, default=str), encoding='utf-8')

if graph_payload is not None:
    gp_path = artifact_dir / 'yfiles_graph_payload.json'
    gp_path.write_text(json.dumps(graph_payload, indent=2, ensure_ascii=False, default=str), encoding='utf-8')
else:
    gp_path = None

show({
    'run_id': run_id,
    'trace_path': str(trace_path),
    'graph_payload_path': str(gp_path) if gp_path else None,
    'components_saved': list(trace['components'].keys()),
})

<IPython.core.display.JSON object>